# Uso de Herramientas

## ¿Qué son las herramientas?
* Algunos casos de uso requieren que los Modelos interactúen directamente con herramientas: sistemas externos—como APIs, bases de datos o sistemas de archivos—utilizando entradas estructuradas.
* Algunos Modelos (p. ej., OpenAI, Anthropic y Gemini) cuentan con herramientas integradas que se ejecutan del lado del servidor, como búsqueda web e intérpretes de código. Consultad la página de [integraciones](https://docs.langchain.com/oss/python/integrations) para más información al respecto.
* También podéis crear herramientas personalizadas.
* Las herramientas amplían las capacidades del Modelo permitiéndole interactuar con el mundo a través de entradas y salidas bien definidas.
* Las Aplicaciones LLM Agénticas (también conocidas como Agentes) pueden decidir si necesitan usar una herramienta disponible para alcanzar un objetivo.

## Como siempre, primero carguemos el archivo .env
* Recordad que en el archivo .env tenemos nuestra clave API de OpenAI y la clave API de Tavily (esto nos permitirá usar la herramienta más adelante).

In [ ]:
from dotenv import load_dotenv

load_dotenv()

#### Recordad cómo podéis obtener vuestra clave API de Tavily

Aquí tenéis cómo obtener vuestra clave API de Tavily en pasos sencillos:

**1. Id al sitio web de Tavily**
Visitad [tavily.com](https://tavily.com) y haced clic en "Get API Key" o "Sign Up"

**2. Cread una cuenta**
Registraos usando vuestra dirección de correo electrónico o mediante un login social (Google, GitHub, etc.)

**3. Verificad vuestro correo electrónico**
Revisad vuestra bandeja de entrada para encontrar un correo de verificación y haced clic en el enlace para confirmar vuestra cuenta

**4. Obtened vuestra clave API gratuita**
Una vez que hayáis iniciado sesión, veréis vuestra clave API en el panel de control. Tavily ofrece un nivel gratuito que incluye 1.000 llamadas API al mes, que es más que suficiente para aprender y hacer los ejercicios del curso

**5. Copiad y guardad la clave**
Copiad la clave API (se parece a una larga cadena de letras y números). Guardadla en un lugar seguro - la necesitaréis para vuestros ejercicios de LangChain

**6. Añadidla en vuestro archivo .env**
```
TAVILY_API_KEY=vuestra-clave-api-aqui
```

**Consejo importante:** Nunca compartáis vuestra clave API públicamente ni la subáis a GitHub. El nivel gratuito debería ser suficiente para la mayoría de los ejercicios del curso, pero siempre podéis consultar vuestro uso en el panel de control de Tavily.

## Primero, probemos un "Agente" sin herramientas

In [ ]:
from langchain.agents import create_agent

system_prompt = "Eres un asistente de pronóstico del tiempo."

agent = create_agent(
    model="gpt-4o-mini",
    system_prompt=system_prompt
)

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="¿Cómo está el tiempo hoy (3 de enero de 2026) en San Francisco?")]}
)

print(response['messages'][-1].content)

## Luego, probemos un Agente muy básico usando una herramienta externa de búsqueda en línea (Tavily)

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Buscar información en la web"""

    return tavily_client.search(query)

#probando la herramienta
web_search.invoke("¿Cómo está el tiempo hoy (3 de enero de 2026) en San Francisco?")

In [ ]:
from langchain.agents import create_agent

weather_agent = create_agent(
    model="gpt-4o-mini",
    tools=[web_search]
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="¿Cómo está el tiempo hoy (3 de enero de 2026) en San Francisco?")

response = weather_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response['messages'])

In [ ]:
print(response["messages"][1].tool_calls)

## Expliquemos el código anterior en términos sencillos

A continuación se muestra el mismo código, explicado **en términos sencillos, línea por línea**, como si fueseis nuevos en Python + LangChain.

---

#### 1) Cargar variables de entorno (claves API, secretos)

```python
from dotenv import load_dotenv
```

* Importa una función auxiliar del paquete `python-dotenv`.
* Este paquete os permite almacenar secretos (como claves API) en un archivo `.env`.

```python
load_dotenv()
```

* Lee vuestro archivo `.env` y carga sus valores en "variables de entorno".
* Ejemplo: si vuestro `.env` contiene `TAVILY_API_KEY=...`, vuestro programa ahora puede acceder a ella automáticamente.

---

#### 2) Importaciones para crear una herramienta de LangChain + tipado

```python
from langchain.tools import tool
```

* Importa el decorador `@tool`.
* Una "herramienta" en LangChain es básicamente una función normal de Python que el agente puede llamar.

```python
from typing import Dict, Any
```

* Importa sugerencias de tipo:

  * `Dict[str, Any]` significa "un diccionario con claves de cadena y valores de cualquier tipo".
* Las sugerencias de tipo ayudan a los humanos (y editores) a entender qué devuelve la función.

```python
from tavily import TavilyClient
```

* Importa el cliente Python de Tavily.
* Tavily es un servicio de API de búsqueda web.

---

#### 3) Crear un cliente Tavily (el objeto que realiza búsquedas)

```python
tavily_client = TavilyClient()
```

* Crea una instancia de cliente Tavily.
* Entre bastidores, este cliente lee vuestra clave API de Tavily de las variables de entorno (por eso cargasteis `.env` antes).

---

#### 4) Definir una función herramienta de LangChain: `web_search`

```python
@tool
def web_search(query: str) -> Dict[str, Any]:
```

* `@tool` convierte esta función de Python en una **herramienta de LangChain**.
* El agente puede llamarla más tarde.
* La función toma una entrada:

  * `query: str` → una cadena de búsqueda
* Devuelve:

  * `Dict[str, Any]` → resultados de búsqueda en forma de diccionario

```python
    """Buscar información en la web"""
```

* Esta es la docstring de la función (una descripción).
* Los agentes a veces pueden leer esta descripción para decidir cuándo usar la herramienta.

```python
    return tavily_client.search(query)
```

* Realmente realiza una búsqueda web usando Tavily.
* Lo que sea que Tavily devuelva (normalmente un diccionario con resultados) es devuelto por vuestra herramienta.

---

## 5) Probar la herramienta directamente (sin un agente)

```python
#probando la herramienta
web_search.invoke("¿Cómo está el tiempo hoy (3 de enero de 2026) en San Francisco?")
```

* Esto llama a la herramienta **manualmente**.
* `invoke(...)` es la forma estándar de LangChain de ejecutar una herramienta.
* Esto es solo para confirmar "la herramienta funciona" antes de dársela a un agente.

---

## 6) Crear un agente y darle vuestra herramienta

```python
from langchain.agents import create_agent
```

* Importa una función auxiliar que construye un "agente".
* Un **agente** es una aplicación LLM que puede decidir:

  * "Debería responder directamente" O
  * "Debería llamar a una herramienta para obtener información"

```python
weather_agent = create_agent(
    model="gpt-4o-mini",
    tools=[web_search]
)
```

* Crea un agente llamado `weather_agent`.
* `model="gpt-4o-mini"`: elige qué LLM usar.
* `tools=[web_search]`: da permiso al agente para llamar a vuestra herramienta `web_search`.
* Así que el agente puede hacer cosas como: "Para responder cómo está el tiempo hoy en una ubicación particular, debería buscar en la web." (Recordad: como vimos con el Agente anterior, el LLM no puede responder a esta pregunta directamente ya que fue entrenado con datos pasados, no datos de hoy. Por eso necesitará Tavily para eso).

---

## 7) Crear un mensaje de usuario

```python
from langchain.messages import HumanMessage
```

* Importa el "objeto mensaje" que representa algo que dice un humano/usuario.

```python
question = HumanMessage(content="¿Cómo está el tiempo hoy (3 de enero de 2026) en San Francisco?")
```

* Construye un mensaje humano con vuestra pregunta dentro.
* Los agentes a menudo trabajan con "mensajes" tipo chat en lugar de cadenas simples.

---

## 8) Hacer la pregunta al agente (el agente puede llamar herramientas)

```python
response = weather_agent.invoke(
    {"messages": [question]}
)
```

* Ejecuta el agente.
* Le pasáis un diccionario con `"messages"` que contiene la conversación hasta el momento.
* El agente:

  1. Leerá la pregunta
  2. Decidirá si necesita la web
  3. Potencialmente llamará a `web_search(...)`
  4. Producirá un mensaje de respuesta final

El resultado (`response`) es típicamente un diccionario que contiene una lista `"messages"` (la conversación incluyendo llamadas a herramientas + respuesta final).

---

## 9) Imprimir el texto de respuesta final

```python
print(response['messages'][-1].content)
```

* `response['messages']` es una lista de mensajes.
* `[-1]` significa "el último mensaje".
* `.content` es el texto de ese mensaje.
* Así que esto imprime la respuesta final del agente.

---

## 10) Imprimir con formato bonito el historial completo de mensajes

```python
from pprint import pprint
```

* Importa una función de impresión más bonita para objetos Python complejos.

```python
pprint(response['messages'])
```

* Imprime *todos* los mensajes de forma ordenada.
* Esto a menudo incluye:

  * vuestro HumanMessage
  * solicitud(es) de llamada a herramientas
  * mensaje(s) de salida de herramientas
  * respuesta final de IA

---

## 11) Inspeccionar las llamadas a herramientas que hizo el agente

```python
print(response["messages"][1].tool_calls)
```

* Observa el **segundo mensaje** en la lista (`[1]`).
* A menudo, el mensaje `[1]` es el mensaje del agente donde decidió llamar a herramientas (depende del comportamiento del agente/framework).
* `.tool_calls` muestra qué herramienta(s) intentó llamar, incluyendo argumentos (como la consulta que usó).

---

# Resumen del modelo mental (súper simple)

* Creaste una **herramienta de búsqueda web** (`web_search`) que usa Tavily.
* Creaste un **agente** que puede usar esa herramienta.
* Preguntaste: "¿Cómo está el tiempo?"
* El agente puede decidir: "Necesito la web → llamar a `web_search` → leer resultados → responder."

## Cómo ejecutar este código desde Visual Studio Code
* Abrid Terminal.
* Aseguraos de estar en la carpeta del proyecto.
* Aseguraos de tener el entorno poetry activado.
* Introducid y ejecutad el siguiente comando:
    * `python 006-tools.py` 

## Consejo Avanzado: Las herramientas son más potentes cuando pueden acceder al estado del agente, contexto de ejecución y memoria a largo plazo.
Esto permite a las herramientas tomar decisiones conscientes del contexto, personalizar respuestas y mantener información a través de conversaciones.

Las herramientas pueden acceder a información de ejecución a través del parámetro `ToolRuntime`, que proporciona:
* State (Estado) - Datos mutables que fluyen a través de la ejecución (p. ej., mensajes, contadores, campos personalizados)
* Context (Contexto) - Configuración inmutable como IDs de usuario, detalles de sesión o configuración específica de la aplicación
* Store (Almacén) - Memoria persistente a largo plazo a través de conversaciones
* Stream Writer (Escritor de flujo) - Transmitir actualizaciones personalizadas mientras se ejecutan las herramientas
* Config (Configuración) - RunnableConfig para la ejecución
* Tool Call ID (ID de llamada a herramienta) - ID de la llamada a herramienta actual

## El parámetro `ToolRuntime`
Permitidnos explicar el parámetro `ToolRuntime` en términos sencillos.

#### ¿Qué es `ToolRuntime`?

Pensad en `ToolRuntime` como una **mochila mágica** a la que vuestra herramienta obtiene acceso automáticamente cuando se ejecuta. Esta mochila contiene información y capacidades importantes que vuestra herramienta podría necesitar.


#### ¿Qué hay dentro de la mochila?

`ToolRuntime` da a vuestra herramienta acceso a 6 cosas importantes:

1. **State (Estado)** - Información que cambia a medida que progresa la conversación (como los mensajes en el chat)
2. **Context (Contexto)** - Configuración fija (como quién es el usuario)
3. **Store (Almacén)** - Memoria a largo plazo (como una base de datos para recordar cosas entre conversaciones)
4. **Streaming (Transmisión)** - Capacidad de enviar actualizaciones mientras la herramienta está trabajando
5. **Config (Configuración)** - Ajustes técnicos
6. **Tool Call ID (ID de llamada a herramienta)** - Un identificador único para esta ejecución específica de la herramienta


#### ¿Por qué es útil esto?

Antes de `ToolRuntime`, teníais que usar 4 métodos complicados diferentes para acceder a estas cosas. Ahora, solo necesitáis **un parámetro simple** - mucho más fácil. `ToolRuntime` reemplaza el patrón antiguo de usar anotaciones separadas de
    * InjectedState,
    * InjectedStore,
    * get_runtime,
    * e InjectedToolCallId.

#### Ejemplo 1: Contando Mensajes

```python
from langchain.tools import tool, ToolRuntime

# Acceder al estado actual de la conversación
@tool
def summarize_conversation(
    runtime: ToolRuntime
) -> str:
    """Resumir la conversación hasta ahora."""
    messages = runtime.state["messages"]

    human_msgs = sum(1 for m in messages if m.__class__.__name__ == "HumanMessage")
    ai_msgs = sum(1 for m in messages if m.__class__.__name__ == "AIMessage")
    tool_msgs = sum(1 for m in messages if m.__class__.__name__ == "ToolMessage")

    return f"La conversación tiene {human_msgs} mensajes de usuario, {ai_msgs} respuestas de IA, y {tool_msgs} resultados de herramientas"
```

**Qué está pasando aquí:**
- La herramienta recibe `runtime` automáticamente (no lo pasáis manualmente)
- Busca dentro de `runtime.state` para encontrar todos los mensajes en la conversación
- Cuenta cuántos mensajes vinieron del humano, la IA y las herramientas
- Devuelve un resumen como "La conversación tiene 3 mensajes de usuario, 3 respuestas de IA, y 2 resultados de herramientas"


#### Ejemplo 2: Obteniendo Preferencias del Usuario

```python
# Acceder a campos de estado personalizados
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime  # El parámetro ToolRuntime no es visible para el modelo
) -> str:
    """Obtener un valor de preferencia del usuario."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "No establecido")
```

**Qué está pasando aquí:**
- Esta herramienta toma DOS parámetros: `pref_name` (qué preferencia buscar) y `runtime`
- **Importante**: El modelo de IA solo "ve" `pref_name` - no sabe sobre `runtime`
- `runtime` es proporcionado automáticamente por el sistema entre bastidores
- La herramienta busca en el estado de la conversación las preferencias del usuario
- Si la preferencia existe, devuelve el valor; de lo contrario devuelve "No establecido"


#### Conclusión Clave

`ToolRuntime` es como tener un **ayudante especial** que da a vuestra herramienta acceso a todo el contexto que necesita sin saturar los parámetros visibles de vuestra herramienta. Es proporcionado automáticamente, la IA no lo ve, y hace vuestras herramientas mucho más potentes.

## Ejemplo Básico: usando ToolRuntime para acceder a Contexto Personalizado

In [ ]:
from dataclasses import dataclass

# Aquí es donde creamos el contexto personalizado
@dataclass
class Preferences:
    vehicle: str = "Vespa"
    city: str = "San Francisco"

from langchain.tools import tool, ToolRuntime

# Aquí es donde usamos el contexto personalizado
@tool
def get_preferred_vehicle(runtime: ToolRuntime[Preferences]) -> str:
    """Obtener el vehículo preferido"""
    return runtime.context.vehicle

@tool
def get_preferred_city(runtime: ToolRuntime[Preferences]) -> str:
    """Obtener la ciudad preferida"""
    return runtime.context.city

# Aquí es donde añadimos las herramientas y el contexto personalizado al agente
agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_preferred_vehicle, get_preferred_city],
    context_schema=Preferences
)

response = agent.invoke(
    {"messages": [HumanMessage(content="¿Cuál es el vehículo preferido?")]},
    context=Preferences()
)

print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response)

## Expliquemos el código anterior en términos sencillos

A continuación se muestra una **explicación amigable para principiantes, línea por línea** del código. Explicaremos **qué hace cada parte** y **por qué existe**.

---

## 1. Importando `dataclass`

```python
from dataclasses import dataclass
```

* Esto importa `dataclass`, una función auxiliar de Python que facilita la creación de clases simples usadas para almacenar datos.
* Pensad en una `dataclass` como un **contenedor limpio para variables**.

---

## 2. Creando un contexto personalizado (memoria compartida)

```python
# Aquí es donde creamos el contexto personalizado

@dataclass
class Preferences:
```

* Estáis definiendo un **objeto de contexto personalizado** llamado `Preferences`.
* "Contexto" significa **información a la que el agente y las herramientas pueden acceder mientras se ejecutan**.

---

```python
    vehicle: str = "Vespa"
    city: str = "San Francisco"
```

* Este contexto tiene dos campos:

  * `vehicle` → valor por defecto `"Vespa"`
  * `city` → valor por defecto `"San Francisco"`
* Si no proporcionáis nada más, se usan estos valores por defecto.
* Podéis pensar en esto como un **perfil de usuario u objeto de configuración**.

---

## 3. Importando utilidades de herramientas de LangChain

```python
from langchain.tools import tool, ToolRuntime
```

* `@tool` es un decorador que convierte una función de Python en una **herramienta que un agente puede llamar**.
* `ToolRuntime` es un objeto que LangChain pasa a las herramientas para que puedan:

  * Acceder al contexto
  * Acceder a datos de ejecución (como memoria, configuración, etc.)

---

## 4. Creando una herramienta que lee del contexto

```python
# Aquí es donde usamos el contexto personalizado

@tool
def get_preferred_vehicle(runtime: ToolRuntime[Preferences]) -> str:
```

* Esto define una **herramienta** llamada `get_preferred_vehicle`.
* `@tool` le dice a LangChain:
  👉 "El agente puede llamar a esta función."
* `runtime: ToolRuntime[Preferences]` significa:

  * LangChain pasará un objeto `runtime`
  * Ese runtime **contiene un contexto `Preferences`**

---

```python
    """Obtener el vehículo preferido"""
```

* Esta es una docstring.
* LangChain usa esta descripción para ayudar al LLM a entender **cuándo usar la herramienta**.

---

```python
    return runtime.context.vehicle
```

* `runtime.context` es el objeto `Preferences` actual.
* `.vehicle` lee el campo `vehicle`.
* Esto devuelve `"Vespa"` (a menos que se haya cambiado el contexto).

---

## 5. Segunda herramienta: ciudad preferida

```python
@tool
def get_preferred_city(runtime: ToolRuntime[Preferences]) -> str:
```

* Misma idea que antes, pero para la ciudad.
* También recibe el runtime y el contexto.

---

```python
    """Obtener la ciudad preferida"""
```

* Descripción de la herramienta para el LLM.

---

```python
    return runtime.context.city
```

* Lee `"San Francisco"` del contexto y lo devuelve.

---

## 6. Creando el agente

```python
# Aquí es donde añadimos las herramientas y el contexto personalizado al agente

agent = create_agent(
```

* Esto crea un **agente de IA**.
* Un agente puede:

  * Leer mensajes
  * Decidir qué herramientas llamar
  * Producir una respuesta final

---

```python
    model="gpt-4o-mini",
```

* Esto le dice al agente qué modelo de lenguaje usar.

---

```python
    tools=[get_preferred_vehicle, get_preferred_city],
```

* Estas son las **únicas herramientas que el agente puede usar**.
* El agente puede llamarlas si piensa que ayudan a responder la pregunta.

---

```python
    context_schema=Preferences
```

* Esto es muy importante.
* Le dice a LangChain:

  * "Este agente usa un contexto con forma de `Preferences`"
* LangChain ahora sabe qué campos existen en `runtime.context`.

---

## 7. Ejecutando el agente

```python
response = agent.invoke(
```

* Esto ejecuta el agente.

---

```python
    {"messages": [HumanMessage(content="¿Cuál es el vehículo preferido?")]},
```

* Le enviáis un **mensaje de chat** al agente.
* El usuario está preguntando:
  👉 "¿Cuál es el vehículo preferido?"

---

```python
    context=Preferences()
```

* Proporcionáis una **instancia del contexto**.
* Como no habéis sobrescrito nada:

  * `vehicle = "Vespa"`
  * `city = "San Francisco"`

---

## 8. Imprimiendo la respuesta final

```python
print(response['messages'][-1].content)
```

* `response['messages']` es el historial de la conversación.
* `[-1]` significa "el último mensaje" (la respuesta del agente).
* `.content` es el texto de la respuesta.

---

## 9. Qué pasa internamente (versión simple)

1. El agente recibe la pregunta
   **"¿Cuál es el vehículo preferido?"**
2. El modelo ve que:

   * Hay una herramienta llamada `get_preferred_vehicle`
   * Esa herramienta devuelve exactamente lo que pide la pregunta
3. El agente llama a la herramienta
4. La herramienta lee `runtime.context.vehicle`
5. El agente responde con algo como:

```
El vehículo preferido es una Vespa.
```

---

## 10. Modelo mental de una frase

> **Este código enseña a un agente de IA cómo leer preferencias de usuario estructuradas (contexto) usando herramientas y responder preguntas sobre ellas.**